# 14_multi_agent_sim

Audience: junior researcher

- Challenge path: `challenges/hard/14_multi_agent_sim`
- Source spec: [challenges/hard/14_multi_agent_sim/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/14_multi_agent_sim/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement a program for a multi-agent flocking simulation (boids). The input consists of:
</p>
  <li>An array <code>agents</code> containing <code>N</code> agents, where <code>N</code> is the total number of agents</li>
  <li>Each agent occupies 4 consecutive 32-bit floating point numbers in the array: \([x, y, v_x, v_y]\), where:
      <ul>
          <li>\((x, y)\) represents the agent's position in 2D space</li>
          <li>\((v_x, v_y)\) represents the agent's velocity vector</li>
      </ul>
  </li>
  <li>The total array size is <code>4 * N</code> floats, with agent \(i\)'s data stored at indices <code>[4i, 4i+1, 4i+2, 4i+3]</code></li>
</ul>

<h2>Simulation Rules</h2>
<ol>
  <li>For each agent \(i\), identify all neighbors \(j\) (where \(i \neq j\)) within radius \(r = 5.0\) using:
      \[
      \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} < r
      \]
  </li>
  <li>Compute average velocity of neighboring agents:
      \[
      \vec{v}_{avg} = \begin{cases}
      \frac{1}{|N_i|} \sum_{j \in N_i} \vec{v}_j & \text{if } |N_i| > 0 \\
      \vec{v}_i & \text{if } |N_i| = 0
      \end{cases}
      \]
      where \(N_i\) is the set of neighbors for agent \(i\)
  </li>
  <li>Update velocity:
      \[
      \vec{v}_{new} = \vec{v} + \alpha(\vec{v}_{avg} - \vec{v}), \text{ where } \alpha = 0.05
      \]
  </li>
  <li>Update position:
      \[
      \vec{p}_{new} = \vec{p} + \vec{v}_{new}
      \]
  </li>
</ol>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>agents_next</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: N = 2
agents = [
  0.0, 0.0, 1.0, 0.0,    // Agent 0: [x, y, vx, vy]
  3.0, 4.0, 0.0, -1.0    // Agent 1: [x, y, vx, vy]
]

Output:
agents_next = [
  1.0, 0.0, 1.0, 0.0,    // Agent 0: [x, y, vx, vy]
  3.0, 3.0, 0.0, -1.0    // Agent 1: [x, y, vx, vy]
]
</pre>

<h2>Constraints</h2>
<ul>
<li>1 &le; <code>N</code> &le; 100,000</li>
<li>Each agent's position and velocity components are 32-bit floats</li>

  <li>Performance is measured with <code>N</code> = 10,000</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/14_multi_agent_sim/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(agents: torch.Tensor, agents_next: torch.Tensor, N: int):
    agents_reshaped = agents.view(N, 4)
    positions = agents_reshaped[:, :2]
    velocities = agents_reshaped[:, 2:]

    diff = positions.unsqueeze(1) - positions.unsqueeze(0)
    dist_sq = (diff**2).sum(dim=2)
    dist_sq.fill_diagonal_(26.0)

    neighbor_mask = dist_sq < 25.0
    sum_velocities = neighbor_mask.float() @ velocities
    neighbor_counts = neighbor_mask.sum(dim=1, keepdim=True)

    avg_velocities = torch.empty_like(velocities)
    nonzero_mask = neighbor_counts[:, 0] > 0
    avg_velocities[nonzero_mask] = sum_velocities[nonzero_mask] / neighbor_counts[nonzero_mask]
    avg_velocities[~nonzero_mask] = velocities[~nonzero_mask]

    new_velocities = velocities + 0.05 * (avg_velocities - velocities)
    new_positions = positions + new_velocities
    agents_next.view(N, 4).copy_(torch.cat([new_positions, new_velocities], dim=1))


## Jax

Source: `challenges/hard/14_multi_agent_sim/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

from typing import Any

try:
    import jax
    import jax.numpy as jnp
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None

import torch


def _solve_torch(agents: torch.Tensor, N: int) -> torch.Tensor:
    agents_reshaped = agents.view(N, 4)
    positions = agents_reshaped[:, :2]
    velocities = agents_reshaped[:, 2:]

    diff = positions.unsqueeze(1) - positions.unsqueeze(0)
    dist_sq = (diff**2).sum(dim=2)
    dist_sq.fill_diagonal_(26.0)

    neighbor_mask = dist_sq < 25.0
    sum_velocities = neighbor_mask.float() @ velocities
    neighbor_counts = neighbor_mask.sum(dim=1, keepdim=True)

    avg_velocities = torch.empty_like(velocities)
    nonzero_mask = neighbor_counts[:, 0] > 0
    avg_velocities[nonzero_mask] = sum_velocities[nonzero_mask] / neighbor_counts[nonzero_mask]
    avg_velocities[~nonzero_mask] = velocities[~nonzero_mask]

    new_velocities = velocities + 0.05 * (avg_velocities - velocities)
    new_positions = positions + new_velocities
    return torch.cat([new_positions, new_velocities], dim=1).reshape(-1)


def _solve_jax(agents: Any, N: int):
    agents_reshaped = jnp.reshape(agents, (N, 4))
    positions = agents_reshaped[:, :2]
    velocities = agents_reshaped[:, 2:]

    diff = positions[:, None, :] - positions[None, :, :]
    dist_sq = jnp.sum(diff**2, axis=2)
    idx = jnp.arange(N)
    dist_sq = dist_sq.at[idx, idx].set(26.0)

    neighbor_mask = dist_sq < 25.0
    sum_velocities = neighbor_mask.astype(velocities.dtype) @ velocities
    neighbor_counts = jnp.sum(neighbor_mask, axis=1, keepdims=True)
    safe_counts = jnp.maximum(neighbor_counts.astype(velocities.dtype), 1.0)
    avg_velocities = sum_velocities / safe_counts
    avg_velocities = jnp.where(neighbor_counts > 0, avg_velocities, velocities)

    new_velocities = velocities + 0.05 * (avg_velocities - velocities)
    new_positions = positions + new_velocities
    return jnp.reshape(jnp.concatenate([new_positions, new_velocities], axis=1), (-1,))


if jax is not None:
    solve_jax = jax.jit(_solve_jax, static_argnames=("N",))

    def solve(agents: Any, N: int, agents_next: Any | None = None):
        result = solve_jax(agents, N)
        if agents_next is not None and hasattr(agents_next, "copy_"):
            agents_next.copy_(torch.as_tensor(result))
            return agents_next
        return result
else:

    def solve(agents: Any, N: int, agents_next: Any | None = None):
        result = _solve_torch(agents, N)
        if agents_next is not None and hasattr(agents_next, "copy_"):
            agents_next.copy_(result)
            return agents_next
        return result


## Triton

Source: `challenges/hard/14_multi_agent_sim/solution/solution.triton.py`

In [ ]:
from __future__ import annotations

from typing import Any

import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(agents: torch.Tensor, agents_next: torch.Tensor, N: int):
    agents_reshaped = agents.view(N, 4)
    positions = agents_reshaped[:, :2]
    velocities = agents_reshaped[:, 2:]

    diff = positions.unsqueeze(1) - positions.unsqueeze(0)
    dist_sq = (diff**2).sum(dim=2)
    dist_sq.fill_diagonal_(26.0)

    neighbor_mask = dist_sq < 25.0
    sum_velocities = neighbor_mask.float() @ velocities
    neighbor_counts = neighbor_mask.sum(dim=1, keepdim=True)

    avg_velocities = torch.empty_like(velocities)
    nonzero_mask = neighbor_counts[:, 0] > 0
    avg_velocities[nonzero_mask] = sum_velocities[nonzero_mask] / neighbor_counts[nonzero_mask]
    avg_velocities[~nonzero_mask] = velocities[~nonzero_mask]

    new_velocities = velocities + 0.05 * (avg_velocities - velocities)
    new_positions = positions + new_velocities
    agents_next.view(N, 4).copy_(torch.cat([new_positions, new_velocities], dim=1))


## Mlx

No solution file is present yet.

## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.